In [ ]:
if 'google.colab' in str(get_ipython()):
  !pip install git+https://github.com/FullControlXYZ/fullcontrol --quiet
import lab.fullcontrol.infaxis as fci
import fullcontrol as fc
from math import sin, cos, tau

fc.Point = fci.Point
fc.GcodeControls = fci.GcodeControls
fc.transform = fci.transform
fc.Axis = fci.Axis

In [ ]:
EW = 0.6
EH = 0.3

print_settings = {'extrusion_width': EW,'extrusion_height': EH}
gcode_controls = fc.GcodeControls(
    head_chain = [],
    bed_chain = [fc.Axis(name='C')],
    initialization_data=print_settings 
    )

In [ ]:
r = 50 # radius of the spiral
density = 360 # number of points per revolution
layers = 60 # number of layers in the z direction for first segment
h = EH # height of each layer
z_start = h*0.5 # starting z height


steps = []
steps.append(fc.Printer(print_speed=2160))
for i in range(layers):
    for j in range(density):
        angle = 360*(i+j/density)
        steps.append(fc.Point(x=r*sin(angle/360*tau), y=r*cos(angle/360*tau), z=((i+j/density)*h)+z_start, axes={'C':angle}))

for step in steps:
    if type(step).__name__ == 'Point':
        # color is a gradient from C=0 deg (blue) to C=360 (red)
        step.color = [((step.axes['C']%360)/360), 0, 1-((step.axes['C']%360)/360)]

fig = fc.transform(steps, 'fig', fc.PlotControls(color_type='manual',style='tube', zoom=0.75), show_tips=False)
fig.show()
gcode = fc.transform(steps,'gcode',gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))